# STOMP: Stochastic Trajectory Optimization for Motion Planning
## A Hands-On Tutorial with a 3R Planar Robotic Arm

This notebook provides a complete, from-scratch implementation of the **STOMP** algorithm — a powerful *sampling-based* trajectory optimizer for robotic motion planning.

**What you'll learn:**
1. How STOMP replaces gradients with **stochastic sampling** for trajectory optimization
2. The role of $R^{-1}$ as both the **noise covariance** and a **smoothing operator**
3. How **probability weighting** selects low-cost perturbations
4. The **$M$ matrix** that keeps updates smooth and bounded

**Prerequisites:** Linear algebra, basic probability, familiarity with forward kinematics.

**Reference:** Kalakrishnan et al., *STOMP: Stochastic Trajectory Optimization for Motion Planning*, ICRA 2011.

---
## 1. The Core Idea

Like CHOMP, STOMP formulates motion planning as **continuous optimization** over a trajectory $\xi : [0, 1] \to \mathcal{Q} \subset \mathbb{R}^d$, minimizing:

$$U[\xi] = F_{\text{obs}}[\xi] + \lambda \, F_{\text{smooth}}[\xi]$$

But STOMP takes a fundamentally different approach to optimization:

| | **CHOMP** | **STOMP** |
|---|---|---|
| **Strategy** | Gradient descent | Stochastic sampling |
| **Requires** | $\nabla_\xi F_{\text{obs}}$ (cost gradient) | $F_{\text{obs}}(\xi)$ (cost evaluation only) |
| **Update** | Step along negative covariant gradient | Weighted average of noisy rollouts |
| **Exploration** | Follows gradient — deterministic | Random perturbations — stochastic |
| **Handles** | Differentiable costs only | **Any** cost function |

**The key insight:** Instead of computing where the gradient points, STOMP generates many *noisy* candidate trajectories, evaluates their costs, and combines the best perturbations via probability weighting. It's an instance of the *path integral* approach to stochastic optimal control.

## 2. Smoothness and the $R$ Matrix

The smoothness cost is identical to CHOMP's:

$$F_{\text{smooth}}[\xi] = \frac{1}{2} \int_0^1 \left\| \ddot{\xi}(t) \right\|^2 dt$$

After discretization with $n$ interior waypoints and finite differences for velocities, we get the $(n+1) \times n$ matrix $K$:

$$K = \begin{pmatrix} 1 & & & \\ -1 & 1 & & \\ & \ddots & \ddots & \\ & & -1 & 1 \\ & & & -1 \end{pmatrix}$$

The **precision matrix** (inverse covariance) is:

$$R = K^\top K$$

This is the same matrix CHOMP calls $A$. Its inverse $R^{-1}$ plays a **dual role** in STOMP:

1. **Covariance for noise sampling:** $\varepsilon \sim \mathcal{N}(0, R^{-1})$ produces smooth perturbations
2. **Smoothing operator:** $R^{-1}$ acts as a low-pass filter, the same "covariant" transformation used in CHOMP

## 3. Stochastic Exploration

At each iteration, STOMP generates $K$ noisy trajectories:

$$\tilde{\theta}_k = \theta + \varepsilon_k, \quad \varepsilon_k \sim \mathcal{N}(0, R^{-1}), \quad k = 1, \ldots, K$$

where $\theta$ is the current trajectory (interior waypoints) and $\varepsilon_k$ is a smooth noise sample.

**Why $R^{-1}$ as covariance?** White noise ($\varepsilon \sim \mathcal{N}(0, I)$) would produce jagged, physically meaningless trajectories. Using $R^{-1}$ as the covariance suppresses high-frequency components, so the perturbations are *inherently smooth*.

**Sampling via Cholesky:** Since $R^{-1} = LL^\top$ (Cholesky decomposition), we generate samples as:
$$\varepsilon = L \cdot z, \quad z \sim \mathcal{N}(0, I)$$

Think of the $K$ noisy trajectories as **scouts** exploring the cost landscape around the current solution. Some scouts will stumble into high-cost regions; others will discover low-cost paths. STOMP learns from all of them.

## 4. Probability Weighting

Each noisy trajectory $\tilde{\theta}_k$ is evaluated at every timestep $i$, producing a per-timestep cost $S(\tilde{\theta}_{k,i})$. STOMP converts these into probabilities:

$$P(k, i) = \frac{e^{-\frac{1}{h} S(\tilde{\theta}_{k,i})}}{\sum_{k'=1}^{K} e^{-\frac{1}{h} S(\tilde{\theta}_{k',i})}}$$

where $h$ is a **temperature** parameter. For numerical stability, we normalize costs before exponentiation:

$$\hat{S}_{k,i} = \frac{S_{k,i} - S_{\min,i}}{S_{\max,i} - S_{\min,i}}$$

| Temperature $h$ | Behavior | Analogy |
|---|---|---|
| $h \to 0$ | Greedy — picks lowest-cost sample | Exploitation only |
| $h \approx 10$ | Balanced — prefers low-cost but considers all | Explore + exploit |
| $h \to \infty$ | Uniform — all samples equally weighted | Exploration only |

The weighted update (per timestep) is:

$$\delta\tilde{\theta}_i = \sum_{k=1}^{K} P(k, i) \cdot \varepsilon_{k,i}$$

## 5. The $M$ Matrix

The probability-weighted update $\delta\tilde{\theta}$ can still be noisy. STOMP applies one more smoothing step using the **$M$ matrix**:

$$\delta\theta = M \cdot \delta\tilde{\theta}$$

$M$ is constructed by **column-normalizing** $R^{-1}$: each column $j$ is scaled so that its maximum absolute value equals $1/n$:

$$M_{:,j} = \frac{R^{-1}_{:,j}}{n \cdot \max_i |R^{-1}_{i,j}|}$$

This achieves two things:
1. **Smoothing:** Since $M$ is derived from $R^{-1}$, it acts as a low-pass filter
2. **Bounded updates:** The normalization ensures the update at each timestep stays within the explored range of the noisy rollouts

## 6. The Cost Function

STOMP uses the same obstacle cost formulation as CHOMP: a **signed distance field** (SDF) with a piecewise cost:

$$c(d) = \begin{cases} -d + \frac{\varepsilon}{2} & d < 0 \\ \frac{1}{2\varepsilon}(d - \varepsilon)^2 & 0 \leq d \leq \varepsilon \\ 0 & d > \varepsilon \end{cases}$$

integrated along the robot body with arc-length weighting: $\sum_{\text{body points}} c(d(x_b)) \cdot \|v_b\|$.

**The critical difference from CHOMP:** STOMP only *evaluates* $c(d)$ — it never differentiates it. This means STOMP can handle:
- Non-differentiable costs (e.g., hard constraint violations)
- Black-box costs (e.g., physics simulators, learned cost functions)
- Costs with discontinuities or noise

## 7. The Complete STOMP Algorithm

$$\boxed{\begin{aligned}
&\textbf{Algorithm: STOMP} \\
&\text{Input: } \theta_0 \text{ (initial trajectory), } K \text{ (samples), } h \text{ (temperature)} \\
&\text{Compute } R = K_{\text{fd}}^\top K_{\text{fd}}, \; R^{-1}, \; M = \text{col-normalize}(R^{-1}), \; L = \text{chol}(R^{-1}) \\[6pt]
&\textbf{repeat} \\
&\quad 1. \; \text{Generate } K \text{ noisy rollouts: } \tilde{\theta}_k = \theta + L \cdot z_k, \; z_k \sim \mathcal{N}(0, I) \\
&\quad 2. \; \text{Evaluate per-timestep costs } S(\tilde{\theta}_{k,i}) \text{ for all } k, i \\
&\quad 3. \; \text{Compute probabilities } P(k,i) = \frac{e^{-\hat{S}_{k,i}/h}}{\sum_{k'} e^{-\hat{S}_{k',i}/h}} \\
&\quad 4. \; \text{Compute weighted update } \delta\tilde{\theta}_i = \sum_k P(k,i) \cdot \varepsilon_{k,i} \\
&\quad 5. \; \text{Smooth: } \delta\theta = M \cdot \delta\tilde{\theta} \\
&\quad 6. \; \text{Update: } \theta \leftarrow \theta + \delta\theta \\
&\textbf{until } \text{convergence}
\end{aligned}}$$

---
# Implementation

Now let's implement everything step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import animation
from matplotlib.colors import Normalize
from scipy.interpolate import RegularGridInterpolator
from IPython.display import HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12
np.random.seed(42)

In [ ]:
# === 3R Planar Arm Parameters ===
L1, L2, L3 = 1.0, 0.8, 0.6       # Link lengths
LINK_LENGTHS = [L1, L2, L3]
DOF = 3                             # Degrees of freedom

# === STOMP Parameters ===
N_WAYPOINTS = 50                    # Number of interior waypoints
K_SAMPLES = 20                      # Number of noisy rollouts per iteration
TEMPERATURE = 10.0                  # Temperature for probability weighting (h)
LAMBDA = 10.0                       # Smoothness weight
MAX_ITER = 100                      # Maximum iterations
EPSILON = 0.3                       # Obstacle safety margin

# === Workspace Parameters ===
WS_XLIM = (-3.0, 3.0)
WS_YLIM = (-3.0, 3.0)
GRID_RESOLUTION = 0.02             # SDF grid cell size
N_BODY_POINTS = 10                 # Body points sampled per link

---
## 8. The 3R Planar Arm

Our robot is a **3-revolute (3R) planar arm** — three rigid links connected by revolute joints, all in the 2D plane. The configuration $q = (\theta_1, \theta_2, \theta_3)$ specifies three joint angles.

In [ ]:
def forward_kinematics(q):
    """
    Compute joint and end-effector positions for the 3R planar arm.
    
    Args:
        q: (3,) array of joint angles [theta1, theta2, theta3]
    Returns:
        positions: (4, 2) array — [base, joint1, joint2, end_effector]
    """
    positions = np.zeros((4, 2))
    cum_angle = 0.0
    for i in range(3):
        cum_angle += q[i]
        positions[i + 1] = positions[i] + LINK_LENGTHS[i] * np.array([np.cos(cum_angle), np.sin(cum_angle)])
    return positions

In [ ]:
def body_point_position(q, link_idx, u):
    """
    Position of a point at parameter u in [0, 1] along link `link_idx`.
    u=0 is the start of the link, u=1 is the end.
    """
    positions = forward_kinematics(q)
    p_start = positions[link_idx]      # start of link
    p_end = positions[link_idx + 1]    # end of link
    return p_start + u * (p_end - p_start)


def compute_jacobian(q, link_idx, u):
    """
    Compute the 2x3 Jacobian for a body point at parameter u on link `link_idx`.
    
    J maps joint velocities dq/dt to workspace velocity dx/dt of this body point.
    """
    J = np.zeros((2, DOF))
    
    # Cumulative angles for each link
    cum_angles = np.cumsum(q)
    
    # Joint j affects this body point only if j <= link_idx
    for j in range(link_idx + 1):
        # Full links from j to link_idx - 1
        for m in range(j, link_idx):
            J[0, j] += -LINK_LENGTHS[m] * np.sin(cum_angles[m])
            J[1, j] +=  LINK_LENGTHS[m] * np.cos(cum_angles[m])
        # Partial link (link_idx) with parameter u
        J[0, j] += -u * LINK_LENGTHS[link_idx] * np.sin(cum_angles[link_idx])
        J[1, j] +=  u * LINK_LENGTHS[link_idx] * np.cos(cum_angles[link_idx])
    
    return J


def compute_all_body_data(q, n_body_points=N_BODY_POINTS):
    """
    Compute all body point positions and Jacobians for a single configuration.
    FK and cumulative angles are computed once, then reused for all body points.
    
    Returns:
        positions: (3 * n_body_points, 2) array of body point positions
        jacobians: (3 * n_body_points, 2, DOF) array of Jacobians
    """
    # FK once
    joint_positions = forward_kinematics(q)
    cum_angles = np.cumsum(q)
    
    n_total = 3 * n_body_points
    positions = np.zeros((n_total, 2))
    jacobians = np.zeros((n_total, 2, DOF))
    
    # Precompute sin/cos of cumulative angles (used by all body points)
    sin_ca = np.sin(cum_angles)
    cos_ca = np.cos(cum_angles)
    
    # Precompute per-link direction vectors (link_start -> link_end)
    link_dirs = np.zeros((3, 2))
    for k in range(3):
        link_dirs[k] = joint_positions[k + 1] - joint_positions[k]
    
    # Precompute cumulative Jacobian contributions for full links
    full_J_contrib = np.zeros((3, DOF, 2))  # [link_idx, joint_j, (x,y)]
    for link_idx in range(3):
        for j in range(link_idx + 1):
            for m in range(j, link_idx):
                full_J_contrib[link_idx, j, 0] += -LINK_LENGTHS[m] * sin_ca[m]
                full_J_contrib[link_idx, j, 1] +=  LINK_LENGTHS[m] * cos_ca[m]
    
    idx = 0
    for link_idx in range(3):
        # Partial link Jacobian contribution (same for all u, just scaled)
        partial_x = -LINK_LENGTHS[link_idx] * sin_ca[link_idx]
        partial_y =  LINK_LENGTHS[link_idx] * cos_ca[link_idx]
        
        for bp in range(n_body_points):
            u = (bp + 0.5) / n_body_points
            
            # Position: interpolate along link
            positions[idx] = joint_positions[link_idx] + u * link_dirs[link_idx]
            
            # Jacobian: full-link contributions + u-scaled partial link
            for j in range(link_idx + 1):
                jacobians[idx, 0, j] = full_J_contrib[link_idx, j, 0] + u * partial_x
                jacobians[idx, 1, j] = full_J_contrib[link_idx, j, 1] + u * partial_y
            
            idx += 1
    
    return positions, jacobians

In [ ]:
def plot_arm(ax, q, color='steelblue', alpha=1.0, linewidth=3, markersize=8, label=None):
    """Draw the 3R arm on the given axes."""
    positions = forward_kinematics(q)
    ax.plot(positions[:, 0], positions[:, 1], 'o-',
            color=color, linewidth=linewidth, markersize=markersize,
            alpha=alpha, solid_capstyle='round', label=label)
    # Mark base with a square
    ax.plot(0, 0, 's', color='black', markersize=10, zorder=5)

In [ ]:
# Visualize the arm in several configurations
test_configs = [
    np.array([0.5, 0.3, 0.2]),
    np.array([1.2, -0.8, 0.5]),
    np.array([np.pi/4, np.pi/3, -np.pi/6]),
    np.array([2.0, -1.0, 0.8]),
]
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

fig, ax = plt.subplots(figsize=(8, 8))
for q_test, c in zip(test_configs, colors):
    plot_arm(ax, q_test, color=c, alpha=0.8,
             label=f'q = ({q_test[0]:.1f}, {q_test[1]:.1f}, {q_test[2]:.1f})')

ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.8, 2.8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
ax.set_title('3R Planar Arm — Test Configurations')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.tight_layout()
plt.show()

---
## 9. Trajectory Representation

We discretize the trajectory into $n$ interior waypoints $\theta_1, \ldots, \theta_n$ (excluding the fixed start $\theta_0$ and goal $\theta_{n+1}$). The decision variable is the $n \times d$ matrix $\xi$, where each row is a waypoint and each column is a joint.

In [ ]:
def init_trajectory(q_start, q_goal, n_waypoints):
    """Create a straight-line trajectory in joint space (excludes endpoints)."""
    xi = np.zeros((n_waypoints, DOF))
    for d in range(DOF):
        xi[:, d] = np.linspace(q_start[d], q_goal[d], n_waypoints + 2)[1:-1]
    return xi


def full_trajectory(xi, q_start, q_goal):
    """Prepend start and append goal to the interior waypoints."""
    return np.vstack([q_start, xi, q_goal])

In [ ]:
def plot_trajectory_workspace(xi, q_start, q_goal, ax=None, obstacles=None,
                              n_arms=8, title='Trajectory in Workspace'):
    """
    Visualize trajectory: draw the arm at several waypoints and trace the end-effector path.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    traj = full_trajectory(xi, q_start, q_goal)
    n_total = len(traj)
    
    # Draw obstacles if any
    if obstacles is not None:
        for obs in obstacles:
            circle = plt.Circle(obs['center'], obs['radius'],
                              color='red', alpha=0.3, zorder=2)
            ax.add_patch(circle)
            circle_edge = plt.Circle(obs['center'], obs['radius'],
                                    fill=False, edgecolor='red', linewidth=2, zorder=2)
            ax.add_patch(circle_edge)
    
    # Draw arm at evenly spaced waypoints
    indices = np.linspace(0, n_total - 1, n_arms, dtype=int)
    for idx in indices:
        alpha = 0.15 + 0.75 * (idx / (n_total - 1))
        plot_arm(ax, traj[idx], color='steelblue', alpha=alpha, linewidth=2, markersize=5)
    
    # Highlight start and goal
    plot_arm(ax, q_start, color='green', linewidth=3, markersize=8, label='Start')
    plot_arm(ax, q_goal, color='red', linewidth=3, markersize=8, label='Goal')
    
    # End-effector path
    ee_path = np.array([forward_kinematics(q)[3] for q in traj])
    ax.plot(ee_path[:, 0], ee_path[:, 1], '-', color='navy', linewidth=2, alpha=0.8, label='EE path')
    
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-2.8, 2.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=10)
    ax.set_title(title)
    return ax


def plot_trajectory_joint_space(xi, q_start, q_goal, title_prefix=''):
    """Plot each joint angle vs. waypoint index."""
    traj = full_trajectory(xi, q_start, q_goal)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for d in range(DOF):
        axes[d].plot(traj[:, d], 'o-', markersize=2, color='steelblue')
        axes[d].set_title(f'{title_prefix}Joint {d+1} ($\theta_{d+1}$)')
        axes[d].set_xlabel('Waypoint index')
        axes[d].set_ylabel('Angle (rad)')
        axes[d].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Define start and goal for our planning problems
q_start = np.array([0.5, 0.3, 0.2])
q_goal = np.array([2.2, -0.8, 1.0])

xi_straight = init_trajectory(q_start, q_goal, N_WAYPOINTS)

fig, ax = plt.subplots(figsize=(8, 8))
plot_trajectory_workspace(xi_straight, q_start, q_goal, ax=ax,
                          title='Initial Straight-Line Trajectory')
plt.tight_layout()
plt.show()

plot_trajectory_joint_space(xi_straight, q_start, q_goal, title_prefix='Initial: ')

---
## 10. The $R$ Matrix, $R^{-1}$, and the $M$ Matrix

We build:
1. **$K$** — the $(n+1) \times n$ finite difference matrix for velocities
2. **$R = K^\top K$** — the $n \times n$ precision matrix (tridiagonal, sparse)
3. **$R^{-1}$** — the covariance matrix (dense, smooth correlations between waypoints)
4. **$M$** — column-normalized $R^{-1}$, used to smooth probability-weighted updates

Note: $R$ is the same matrix CHOMP calls $A$, and $R^{-1}$ is $A^{-1}$.

In [ ]:
def build_stomp_matrices(n_waypoints, q_start, q_goal):
    """
    Build STOMP matrices: K, R, R_inv, M, and boundary vectors e, b.
    
    Returns:
        K: (n+1, n) finite difference matrix
        R: (n, n) precision matrix = K^T K
        R_inv: (n, n) covariance matrix = R^{-1}
        M: (n, n) column-normalized R_inv for update smoothing
        e: (n+1, DOF) boundary vector
        b: (n, DOF) boundary contribution = K^T e
    """
    n = n_waypoints
    
    # K matrix: (n+1) x n
    K = np.zeros((n + 1, n))
    K[0, 0] = 1.0
    for i in range(1, n):
        K[i, i] = 1.0
        K[i, i - 1] = -1.0
    K[n, n - 1] = -1.0
    
    # Boundary vector e: (n+1) x DOF
    e = np.zeros((n + 1, DOF))
    e[0, :] = -q_start
    e[n, :] = q_goal
    
    # Precision matrix R = K^T K (symmetric positive definite)
    R = K.T @ K
    
    # Covariance matrix R_inv = R^{-1}
    R_inv = np.linalg.inv(R)
    
    # M matrix: column-normalized R_inv
    # Each column j is scaled so max|M[:,j]| = 1/n
    M = R_inv.copy()
    for j in range(n):
        max_abs = np.max(np.abs(M[:, j]))
        if max_abs > 0:
            M[:, j] /= (n * max_abs)
    
    # Boundary contribution to gradient: b = K^T e
    b = K.T @ e
    
    return K, R, R_inv, M, e, b


def compute_smoothness_cost(xi, K, e):
    """F_smooth = (1/2) sum_d ||K @ xi_d + e_d||^2"""
    cost = 0.0
    for d in range(DOF):
        vel = K @ xi[:, d] + e[:, d]
        cost += 0.5 * np.dot(vel, vel)
    return cost


def compute_smoothness_gradient(xi, R, b):
    """Gradient of F_smooth w.r.t. xi: R @ xi + b. Shape: (n, DOF)."""
    return R @ xi + b

In [ ]:
# Build matrices and visualize
K, R, R_inv, M, e, b = build_stomp_matrices(N_WAYPOINTS, q_start, q_goal)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# R (precision)
im0 = axes[0].imshow(R, cmap='RdBu_r', aspect='equal')
axes[0].set_title('$R = K^\\top K$ (Precision)', fontsize=13)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# R_inv (covariance)
im1 = axes[1].imshow(R_inv, cmap='RdBu_r', aspect='equal')
axes[1].set_title('$R^{-1}$ (Covariance)', fontsize=13)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# M (column-normalized)
im2 = axes[2].imshow(M, cmap='RdBu_r', aspect='equal')
axes[2].set_title('$M$ (Column-Normalized $R^{-1}$)', fontsize=13)
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle('STOMP Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'R is tridiagonal (sparse): {np.count_nonzero(np.abs(R) > 1e-10)} non-zero entries out of {R.size}')
print(f'R_inv is dense: {np.count_nonzero(np.abs(R_inv) > 1e-10)} non-zero entries out of {R_inv.size}')
print(f'R_inv correlates distant waypoints — this is why noise from R_inv is smooth.')

### Understanding Noise from $R^{-1}$

The key insight: $R$ is **sparse** (tridiagonal), but $R^{-1}$ is **dense**. This means:

- **White noise** $z \sim \mathcal{N}(0, I)$: each waypoint is independent — the result is jagged
- **Smooth noise** $\varepsilon = L \cdot z$ where $R^{-1} = LL^\top$: waypoints are correlated — the result is smooth

This is the same reason CHOMP uses the "covariant" gradient $A^{-1} \nabla f$ instead of the raw gradient $\nabla f$: multiplying by $R^{-1}$ acts as a low-pass filter.

In [ ]:
def generate_smooth_noise(L_chol, n_waypoints, K_samples):
    """
    Generate K smooth noise samples using Cholesky factor of R_inv.
    
    Args:
        L_chol: (n, n) lower Cholesky factor such that R_inv = L @ L.T
        n_waypoints: number of interior waypoints
        K_samples: number of noise samples
    Returns:
        noise: (K_samples, n_waypoints, DOF) array of smooth noise
    """
    noise = np.zeros((K_samples, n_waypoints, DOF))
    for k in range(K_samples):
        for d in range(DOF):
            z = np.random.randn(n_waypoints)
            noise[k, :, d] = L_chol @ z
    return noise


# Cholesky decomposition of R_inv
L_chol = np.linalg.cholesky(R_inv)

# Generate noise samples for visualization
np.random.seed(42)
smooth_noise = generate_smooth_noise(L_chol, N_WAYPOINTS, 5)
white_noise = np.random.randn(5, N_WAYPOINTS, DOF)

# Compare white noise vs smooth noise
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
waypoints = np.arange(N_WAYPOINTS)
joint_names = [r'$\theta_1$', r'$\theta_2$', r'$\theta_3$']
colors_5 = ['steelblue', 'coral', 'seagreen', 'mediumpurple', 'goldenrod']

for d in range(DOF):
    # White noise (top row)
    for k in range(5):
        axes[0, d].plot(waypoints, white_noise[k, :, d], color=colors_5[k], alpha=0.7, linewidth=1)
    axes[0, d].set_title(f'White Noise — {joint_names[d]}')
    axes[0, d].set_xlabel('Waypoint')
    axes[0, d].set_ylabel('Perturbation')
    axes[0, d].grid(True, alpha=0.3)
    
    # Smooth noise (bottom row)
    for k in range(5):
        axes[1, d].plot(waypoints, smooth_noise[k, :, d], color=colors_5[k], alpha=0.7, linewidth=1.5)
    axes[1, d].set_title(f'$R^{{-1}}$ Noise — {joint_names[d]}')
    axes[1, d].set_xlabel('Waypoint')
    axes[1, d].set_ylabel('Perturbation')
    axes[1, d].grid(True, alpha=0.3)

plt.suptitle('White Noise vs. Smooth ($R^{-1}$) Noise — 5 Samples Each', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('White noise is jagged — each waypoint is independent.')
print('R_inv noise is smooth — waypoints are correlated by the covariance structure.')

## 11. Probability Weighting

The probability of selecting sample $k$ at timestep $i$ uses min/max normalization for numerical stability, followed by softmax:

$$P(k, i) = \frac{e^{-\frac{1}{h}\hat{S}_{k,i}}}{\sum_{k'} e^{-\frac{1}{h}\hat{S}_{k',i}}}, \quad \hat{S}_{k,i} = \frac{S_{k,i} - S_{\min,i}}{S_{\max,i} - S_{\min,i}}$$

In [ ]:
def compute_probabilities(costs, temperature):
    """
    Compute per-timestep probability weights for K samples.
    
    Args:
        costs: (K, N) array of per-timestep costs for each sample
        temperature: scalar h — controls exploitation vs exploration
    Returns:
        probs: (K, N) array of probability weights (columns sum to 1)
    """
    # Min/max normalization per timestep for numerical stability
    S_min = costs.min(axis=0)   # (N,)
    S_max = costs.max(axis=0)   # (N,)
    denom = S_max - S_min + 1e-10
    normalized = (costs - S_min) / denom  # (K, N) in [0, 1]
    
    # Softmax with temperature
    exp_costs = np.exp(-normalized / temperature)  # (K, N)
    probs = exp_costs / exp_costs.sum(axis=0)      # (K, N), columns sum to 1
    
    return probs

---
## 12. Case 1: STOMP Without Obstacles

As a sanity check, we first run STOMP with only the smoothness cost. Starting from a perturbed trajectory with sinusoidal bumps, STOMP should converge to a straight line in joint space (the minimum-velocity trajectory).

In [ ]:
# Create a perturbed initial trajectory with sinusoidal bumps
xi_perturbed = init_trajectory(q_start, q_goal, N_WAYPOINTS).copy()
t = np.linspace(0, 1, N_WAYPOINTS)
xi_perturbed[:, 0] += 0.5 * np.sin(2 * np.pi * t)
xi_perturbed[:, 1] -= 0.4 * np.sin(4 * np.pi * t)
xi_perturbed[:, 2] += 0.3 * np.sin(3 * np.pi * t)

In [ ]:
def compute_state_costs_smoothness_only(xi, K_mat, e_vec):
    """
    Compute per-waypoint smoothness cost for STOMP.
    
    For each waypoint i, compute its contribution to the total squared velocity cost.
    This gives STOMP a per-timestep signal for probability weighting.
    
    Args:
        xi: (n, DOF) trajectory waypoints
        K_mat: (n+1, n) finite difference matrix
        e_vec: (n+1, DOF) boundary vector
    Returns:
        costs: (n,) per-waypoint smoothness costs
    """
    n = xi.shape[0]
    costs = np.zeros(n)
    
    for d in range(DOF):
        vel = K_mat @ xi[:, d] + e_vec[:, d]  # (n+1,) velocity differences
        # Distribute velocity cost to the waypoints that contribute to each velocity
        for i in range(n):
            # vel[i] involves waypoint i (as the "to" endpoint)
            costs[i] += 0.5 * vel[i]**2
    
    return costs

In [ ]:
def stomp_no_obstacles(xi_init, q_start, q_goal, n_waypoints,
                       K_samples=K_SAMPLES, temperature=TEMPERATURE,
                       lam=LAMBDA, max_iter=MAX_ITER):
    """
    STOMP with only smoothness cost.
    
    Steps per iteration:
    1. Generate K noisy rollouts via Cholesky
    2. Evaluate per-waypoint smoothness cost for each rollout
    3. Compute probability weights
    4. Compute weighted update
    5. Smooth with M matrix
    6. Update trajectory
    """
    K_mat, R, R_inv, M_mat, e_vec, b_vec = build_stomp_matrices(n_waypoints, q_start, q_goal)
    L_chol_local = np.linalg.cholesky(R_inv)
    
    xi = xi_init.copy()
    cost_history = []
    trajectory_history = [xi.copy()]
    
    for iteration in range(max_iter):
        # Record total smoothness cost
        cost = compute_smoothness_cost(xi, K_mat, e_vec)
        cost_history.append(cost)
        
        # 1. Generate K noisy rollouts
        noise = generate_smooth_noise(L_chol_local, n_waypoints, K_samples)  # (K, n, DOF)
        
        # 2. Evaluate per-waypoint costs for each rollout
        sample_costs = np.zeros((K_samples, n_waypoints))  # (K, n)
        for k in range(K_samples):
            xi_noisy = xi + noise[k]
            sample_costs[k] = compute_state_costs_smoothness_only(xi_noisy, K_mat, e_vec)
        
        # 3. Probability weights
        probs = compute_probabilities(sample_costs, temperature)  # (K, n)
        
        # 4. Weighted update per DOF
        delta_tilde = np.zeros_like(xi)  # (n, DOF)
        for d in range(DOF):
            # noise[:, :, d] is (K, n), probs is (K, n)
            delta_tilde[:, d] = np.sum(probs * noise[:, :, d], axis=0)
        
        # 5. Smooth with M matrix
        delta = np.zeros_like(xi)
        for d in range(DOF):
            delta[:, d] = M_mat @ delta_tilde[:, d]
        
        # 6. Update
        xi = xi + delta
        
        if iteration % 10 == 0:
            trajectory_history.append(xi.copy())
    
    trajectory_history.append(xi.copy())
    return xi, cost_history, trajectory_history

In [ ]:
np.random.seed(42)
xi_smooth, costs_smooth, traj_hist_smooth = stomp_no_obstacles(
    xi_perturbed, q_start, q_goal, N_WAYPOINTS)

# Cost convergence
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(costs_smooth, 'steelblue', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Smoothness Cost $F_{smooth}$')
ax.set_title('STOMP Without Obstacles: Cost Convergence')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Initial smoothness cost: {costs_smooth[0]:.4f}')
print(f'Final smoothness cost:   {costs_smooth[-1]:.4f}')

In [ ]:
# Workspace comparison: initial vs optimized vs evolution
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

plot_trajectory_workspace(xi_perturbed, q_start, q_goal, ax=axes[0],
                          title='Initial (Perturbed)')
plot_trajectory_workspace(xi_smooth, q_start, q_goal, ax=axes[1],
                          title='Optimized (Smooth)')

# Evolution overlay — end-effector paths from intermediate trajectories
ax = axes[2]
plot_arm(ax, q_start, color='green', linewidth=3, label='Start')
plot_arm(ax, q_goal, color='red', linewidth=3, label='Goal')
cmap = plt.cm.viridis
for j, xi_hist in enumerate(traj_hist_smooth):
    traj = full_trajectory(xi_hist, q_start, q_goal)
    ee = np.array([forward_kinematics(q)[3] for q in traj])
    color = cmap(j / max(len(traj_hist_smooth) - 1, 1))
    ax.plot(ee[:, 0], ee[:, 1], '-', color=color, alpha=0.6, linewidth=1.5)
ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.8, 2.8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
ax.set_title('Evolution (dark = early, light = late)')

plt.tight_layout()
plt.show()

In [ ]:
# Joint space: before vs after
traj_init = full_trajectory(xi_perturbed, q_start, q_goal)
traj_opt = full_trajectory(xi_smooth, q_start, q_goal)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for d in range(DOF):
    axes[d].plot(traj_init[:, d], '--', color='coral', linewidth=2, label='Initial')
    axes[d].plot(traj_opt[:, d], '-', color='steelblue', linewidth=2, label='Optimized')
    axes[d].set_title(f'Joint {d+1} ($\theta_{d+1}$)')
    axes[d].set_xlabel('Waypoint index')
    axes[d].set_ylabel('Angle (rad)')
    axes[d].grid(True, alpha=0.3)
    axes[d].legend(fontsize=9)
plt.suptitle('Joint Space: Bumpy Initial → Smooth Straight Line', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Without obstacles, STOMP drives the trajectory to a straight line in joint space.')
print(f'Final smoothness cost: {costs_smooth[-1]:.4f}')

---
## 13. Obstacle Cost

We use the same SDF-based obstacle cost as CHOMP. The key difference: **STOMP only evaluates $c(d)$, never differentiates it.**

In [ ]:
# Define obstacles (circular)
obstacles = [
    {'center': np.array([1.2, 1.0]),  'radius': 0.35},
    {'center': np.array([0.0, 1.4]),  'radius': 0.3},
    {'center': np.array([-0.5, 0.8]), 'radius': 0.3},
]

In [ ]:
def compute_sdf(obstacles, xlim=WS_XLIM, ylim=WS_YLIM, resolution=GRID_RESOLUTION):
    """
    Compute signed distance field on a grid.
    For circular obstacles: d_i(x) = ||x - c_i|| - r_i.
    Composite: d(x) = min_i d_i(x).
    
    Returns: sdf, sdf_grad_x, sdf_grad_y, x_coords, y_coords
    """
    x_coords = np.arange(xlim[0], xlim[1] + resolution, resolution)
    y_coords = np.arange(ylim[0], ylim[1] + resolution, resolution)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    sdf = np.full_like(X, np.inf)
    for obs in obstacles:
        dist = np.sqrt((X - obs['center'][0])**2 + (Y - obs['center'][1])**2) - obs['radius']
        sdf = np.minimum(sdf, dist)
    
    # Gradient via central differences on the composite SDF
    sdf_grad_y, sdf_grad_x = np.gradient(sdf, resolution)
    
    return sdf, sdf_grad_x, sdf_grad_y, x_coords, y_coords

In [ ]:
# Obstacle cost function — STOMP does not need cost gradients
def obstacle_cost_vectorized(d, epsilon=EPSILON):
    """Piecewise cost: large inside obstacles, decaying in safety margin, zero outside."""
    cost = np.zeros_like(d)
    mask_inside = d < 0
    mask_margin = (d >= 0) & (d <= epsilon)
    cost[mask_inside] = -d[mask_inside] + epsilon / 2.0
    cost[mask_margin] = (d[mask_margin] - epsilon)**2 / (2.0 * epsilon)
    return cost


def obstacle_cost_scalar(d, epsilon=EPSILON):
    """Scalar version for single-point evaluation."""
    if d < 0:
        return -d + epsilon / 2.0
    elif d <= epsilon:
        return (d - epsilon)**2 / (2.0 * epsilon)
    return 0.0

In [ ]:
# Visualize SDF and cost field
sdf, sdf_gx, sdf_gy, x_coords, y_coords = compute_sdf(obstacles)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# SDF
ax = axes[0]
levels = np.linspace(-1, 2, 30)
cf = ax.contourf(x_coords, y_coords, sdf, levels=levels, cmap='RdYlBu', extend='both')
ax.contour(x_coords, y_coords, sdf, levels=[0], colors='black', linewidths=2)
ax.contour(x_coords, y_coords, sdf, levels=[EPSILON], colors='orange', linewidths=1.5, linestyles='--')
plt.colorbar(cf, ax=ax, shrink=0.8, label='Signed distance $d(x)$')
ax.set_title('Signed Distance Field (SDF)', fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
ax.set_xlim(-2, 2.5); ax.set_ylim(-1, 2.5)

# Cost field c(d)
ax = axes[1]
cost_field = obstacle_cost_vectorized(sdf)
cf2 = ax.contourf(x_coords, y_coords, cost_field, levels=20, cmap='hot_r')
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(circle)
plt.colorbar(cf2, ax=ax, shrink=0.8, label='Cost $c(d(x))$')
ax.set_title(f'Obstacle Cost Field ($\varepsilon$ = {EPSILON})', fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
ax.set_xlim(-2, 2.5); ax.set_ylim(-1, 2.5)

plt.tight_layout()
plt.show()

In [ ]:
def build_sdf_interpolators(sdf, x_coords, y_coords):
    """Build scipy interpolation function for SDF only (STOMP needs no gradient)."""
    sdf_interp = RegularGridInterpolator(
        (y_coords, x_coords), sdf,
        method='linear', bounds_error=False, fill_value=10.0)
    return sdf_interp

In [ ]:
def compute_obstacle_state_costs(xi, q_start, q_goal, sdf_interp,
                                 n_body_points=N_BODY_POINTS, epsilon=EPSILON):
    """
    Compute per-waypoint obstacle costs (no gradients needed for STOMP).
    
    For each waypoint: FK -> body points -> SDF query -> c(d) -> arc-length weight -> sum.
    
    Args:
        xi: (n, DOF) trajectory waypoints
        sdf_interp: SDF interpolation function
    Returns:
        costs: (n,) per-waypoint obstacle costs
    """
    n = xi.shape[0]
    costs = np.zeros(n)
    n_bp_total = 3 * n_body_points
    
    traj = full_trajectory(xi, q_start, q_goal)  # (n+2, DOF)
    
    for i in range(n):
        q = traj[i + 1]
        q_vel = (traj[i + 2] - traj[i]) / 2.0
        
        # Compute all body point positions and Jacobians
        bp_positions, bp_jacobians = compute_all_body_data(q, n_body_points)
        
        # Batch SDF queries (y, x order)
        query_pts = bp_positions[:, ::-1]
        d_vals = sdf_interp(query_pts)
        
        # Cost values (no gradient needed)
        c_vals = obstacle_cost_vectorized(d_vals, epsilon)
        
        # Velocity magnitudes for arc-length weighting
        v_bps = bp_jacobians @ q_vel       # (n_bp_total, 2)
        vel_mags = np.linalg.norm(v_bps, axis=1) + 1e-8
        
        # Weighted sum of costs at this waypoint
        costs[i] = np.sum(c_vals * vel_mags) / n_body_points
    
    return costs

In [ ]:
# Build interpolator
sdf_interp = build_sdf_interpolators(sdf, x_coords, y_coords)

# Visualize: arm near obstacles with body point costs
q_demo = np.array([0.8, 0.4, 0.2])  # config near obstacles

fig, ax = plt.subplots(figsize=(8, 8))
# Cost field background
cost_field = obstacle_cost_vectorized(sdf)
ax.contourf(x_coords, y_coords, cost_field, levels=20, cmap='hot_r', alpha=0.4)
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.3)
    ax.add_patch(circle)
    circle_edge = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='red', linewidth=2)
    ax.add_patch(circle_edge)

# Draw arm
plot_arm(ax, q_demo, color='steelblue', linewidth=3)

# Body points colored by cost
for link_idx in range(3):
    for bp in range(N_BODY_POINTS):
        u = (bp + 0.5) / N_BODY_POINTS
        x_bp = body_point_position(q_demo, link_idx, u)
        d_val = sdf_interp(np.array([[x_bp[1], x_bp[0]]])).item()
        c_val = obstacle_cost_scalar(d_val)
        color = 'yellow' if c_val > 0.01 else 'lightblue'
        size = 30 + 200 * c_val
        ax.scatter(x_bp[0], x_bp[1], c=[c_val], cmap='hot_r', s=size,
                   vmin=0, vmax=0.5, edgecolors='black', linewidths=0.5, zorder=10)

ax.set_xlim(-1.5, 2.5)
ax.set_ylim(-1, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Body Points Colored by Obstacle Cost')
plt.tight_layout()
plt.show()

---
## 14. Case 2: STOMP With Obstacles

Now we run the full STOMP algorithm with obstacle + smoothness cost:

```
repeat:
    1. Generate K noisy rollouts: theta_k = theta + L*z_k
    2. Evaluate per-timestep costs S(theta_{k,i}) = obstacle_cost_i + lambda*smoothness_cost_i
    3. Compute probabilities P(k,i) via softmax
    4. Weighted update: delta_theta_i = sum_k P(k,i)*epsilon_{k,i}
    5. Smooth: delta_theta = M*delta_theta_tilde
    6. Update: theta <- theta + delta_theta
until convergence
```

In [ ]:
def stomp_with_obstacles(xi_init, q_start, q_goal, obstacles,
                         K_samples=K_SAMPLES, temperature=TEMPERATURE,
                         lam=LAMBDA, max_iter=MAX_ITER,
                         epsilon=EPSILON, n_body_points=N_BODY_POINTS,
                         record_every=5):
    """
    Full STOMP algorithm with smoothness + obstacle cost.
    
    Returns:
        xi_opt: optimized trajectory
        cost_history: dict with 'total', 'smooth', 'obstacle' lists
        trajectory_history: list of trajectory snapshots
    """
    n = xi_init.shape[0]
    
    # Build STOMP matrices
    K_mat, R, R_inv, M_mat, e_vec, b_vec = build_stomp_matrices(n, q_start, q_goal)
    L_chol_local = np.linalg.cholesky(R_inv)
    
    # Build SDF and interpolator
    sdf_data, sdf_gx, sdf_gy, xc, yc = compute_sdf(obstacles)
    si = build_sdf_interpolators(sdf_data, xc, yc)
    
    xi = xi_init.copy()
    cost_history = {'total': [], 'smooth': [], 'obstacle': []}
    trajectory_history = [xi.copy()]
    
    for iteration in range(max_iter):
        # Evaluate current costs for recording
        cost_smooth = compute_smoothness_cost(xi, K_mat, e_vec)
        cost_obs_total = np.sum(compute_obstacle_state_costs(
            xi, q_start, q_goal, si, n_body_points, epsilon))
        cost_history['total'].append(cost_obs_total + lam * cost_smooth)
        cost_history['smooth'].append(cost_smooth)
        cost_history['obstacle'].append(cost_obs_total)
        
        # 1. Generate K noisy rollouts
        noise = generate_smooth_noise(L_chol_local, n, K_samples)  # (K, n, DOF)
        
        # 2. Evaluate per-waypoint costs for each noisy rollout
        sample_costs = np.zeros((K_samples, n))
        for k in range(K_samples):
            xi_noisy = xi + noise[k]
            # Per-waypoint obstacle cost
            obs_costs = compute_obstacle_state_costs(
                xi_noisy, q_start, q_goal, si, n_body_points, epsilon)
            # Per-waypoint smoothness cost
            smooth_costs = compute_state_costs_smoothness_only(xi_noisy, K_mat, e_vec)
            # Combined
            sample_costs[k] = obs_costs + lam * smooth_costs
        
        # 3. Probability weights
        probs = compute_probabilities(sample_costs, temperature)  # (K, n)
        
        # 4. Weighted update per DOF
        delta_tilde = np.zeros_like(xi)
        for d in range(DOF):
            delta_tilde[:, d] = np.sum(probs * noise[:, :, d], axis=0)
        
        # 5. Smooth with M matrix
        delta = np.zeros_like(xi)
        for d in range(DOF):
            delta[:, d] = M_mat @ delta_tilde[:, d]
        
        # 6. Update
        xi = xi + delta
        
        if iteration % record_every == 0:
            trajectory_history.append(xi.copy())
    
    trajectory_history.append(xi.copy())
    return xi, cost_history, trajectory_history

In [ ]:
# Run STOMP with obstacles
xi_init_obs = init_trajectory(q_start, q_goal, N_WAYPOINTS)

np.random.seed(42)
print('Running STOMP with obstacles...')
xi_opt_obs, costs_obs, traj_hist_obs = stomp_with_obstacles(
    xi_init_obs, q_start, q_goal, obstacles,
    K_samples=K_SAMPLES, temperature=TEMPERATURE, lam=LAMBDA, max_iter=MAX_ITER)
print(f'Done. Final total cost: {costs_obs["total"][-1]:.4f}')
print(f'  Obstacle cost: {costs_obs["obstacle"][-1]:.4f}')
print(f'  Smoothness cost: {costs_obs["smooth"][-1]:.4f}')

In [ ]:
# Cost convergence (3 panels)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = ['Total Cost $U[\\xi]$', 'Obstacle Cost $F_{obs}$', 'Smoothness Cost $F_{smooth}$']
keys = ['total', 'obstacle', 'smooth']
colors_plot = ['navy', 'red', 'steelblue']

for ax, title, key, color in zip(axes, titles, keys, colors_plot):
    ax.plot(costs_obs[key], color=color, linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cost')
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    if min(costs_obs[key]) > 0:
        ax.set_yscale('log')

plt.suptitle('STOMP With Obstacles: Cost Convergence', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Before vs After (static comparison)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

plot_trajectory_workspace(xi_init_obs, q_start, q_goal, ax=axes[0],
                          obstacles=obstacles,
                          title='Initial Trajectory (Straight Line)')
plot_trajectory_workspace(xi_opt_obs, q_start, q_goal, ax=axes[1],
                          obstacles=obstacles,
                          title='Optimized Trajectory (STOMP)')

plt.tight_layout()
plt.show()

In [ ]:
# Summary figure: cost field + optimized trajectory
fig, ax = plt.subplots(figsize=(10, 10))

# Cost field background
cost_field = obstacle_cost_vectorized(sdf)
ax.contourf(x_coords, y_coords, cost_field, levels=20, cmap='hot_r', alpha=0.3)

# Obstacles
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.4)
    ax.add_patch(circle)
    circle_edge = plt.Circle(obs['center'], obs['radius'],
                            fill=False, edgecolor='darkred', linewidth=2)
    ax.add_patch(circle_edge)

# Draw optimized arm at several waypoints
traj_full = full_trajectory(xi_opt_obs, q_start, q_goal)
n_show = 10
show_idx = np.linspace(0, len(traj_full) - 1, n_show, dtype=int)
for idx in show_idx:
    alpha = 0.15 + 0.7 * (idx / (len(traj_full) - 1))
    plot_arm(ax, traj_full[idx], color='steelblue', alpha=alpha, linewidth=2, markersize=4)

# Start and goal
plot_arm(ax, q_start, color='green', linewidth=3.5, markersize=9, label='Start')
plot_arm(ax, q_goal, color='red', linewidth=3.5, markersize=9, label='Goal')

# End-effector path
ee_path = np.array([forward_kinematics(q)[3] for q in traj_full])
ax.plot(ee_path[:, 0], ee_path[:, 1], '-', color='navy', linewidth=2.5, alpha=0.9, label='EE path')

ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.0, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=12)
ax.set_title('STOMP Result: Optimized Trajectory Avoiding Obstacles', fontsize=14)
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

In [ ]:
# Animation: trajectory evolution
def animate_stomp(traj_hist, q_start, q_goal, obstacles, interval=300):
    """Animate STOMP trajectory evolution."""
    fig, ax = plt.subplots(figsize=(9, 9))
    
    # Precompute cost field
    sdf_local, _, _, xc_local, yc_local = compute_sdf(obstacles)
    cost_field_local = obstacle_cost_vectorized(sdf_local)
    
    def draw_frame(frame_idx):
        ax.clear()
        
        # Cost field
        ax.contourf(xc_local, yc_local, cost_field_local, levels=20, cmap='hot_r', alpha=0.25)
        
        # Obstacles
        for obs in obstacles:
            circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.4)
            ax.add_patch(circle)
            ce = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='darkred', linewidth=2)
            ax.add_patch(ce)
        
        xi_frame = traj_hist[frame_idx]
        traj_f = full_trajectory(xi_frame, q_start, q_goal)
        
        # Arms at waypoints
        n_arms = 8
        idx_show = np.linspace(0, len(traj_f) - 1, n_arms, dtype=int)
        for idx in idx_show:
            alpha = 0.15 + 0.65 * (idx / (len(traj_f) - 1))
            plot_arm(ax, traj_f[idx], color='steelblue', alpha=alpha, linewidth=2, markersize=4)
        
        plot_arm(ax, q_start, color='green', linewidth=3, markersize=8)
        plot_arm(ax, q_goal, color='red', linewidth=3, markersize=8)
        
        # EE path
        ee = np.array([forward_kinematics(q)[3] for q in traj_f])
        ax.plot(ee[:, 0], ee[:, 1], '-', color='navy', linewidth=2, alpha=0.8)
        
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.0, 2.5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        iteration = frame_idx * 5  # since we record every 5 iterations
        ax.set_title(f'STOMP Evolution — Iteration {iteration}', fontsize=13)
    
    anim = animation.FuncAnimation(fig, draw_frame, frames=len(traj_hist), interval=interval)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_stomp(traj_hist_obs, q_start, q_goal, obstacles)

## 15. Visualizing Noisy Rollouts

A unique advantage of STOMP: we can visualize **how the algorithm explores**. Each iteration generates $K$ noisy trajectory rollouts. We can plot these rollouts and color them by their probability weight to see which perturbations STOMP favors.

In [ ]:
# Visualize one iteration of STOMP: show all K noisy rollouts
np.random.seed(123)

# Use the optimized trajectory as the current state (show exploration around a good solution)
xi_current = xi_opt_obs.copy()
n = xi_current.shape[0]

# Build matrices
K_mat_viz, R_viz, R_inv_viz, M_viz, e_viz, b_viz = build_stomp_matrices(n, q_start, q_goal)
L_chol_viz = np.linalg.cholesky(R_inv_viz)
sdf_data_viz, _, _, xc_viz, yc_viz = compute_sdf(obstacles)
si_viz = build_sdf_interpolators(sdf_data_viz, xc_viz, yc_viz)

# Generate noisy rollouts
K_viz = 30
noise_viz = generate_smooth_noise(L_chol_viz, n, K_viz)

# Evaluate costs for probability coloring
sample_costs_viz = np.zeros((K_viz, n))
for k in range(K_viz):
    xi_noisy = xi_current + noise_viz[k]
    obs_costs = compute_obstacle_state_costs(xi_noisy, q_start, q_goal, si_viz)
    smooth_costs = compute_state_costs_smoothness_only(xi_noisy, K_mat_viz, e_viz)
    sample_costs_viz[k] = obs_costs + LAMBDA * smooth_costs

# Mean cost per rollout for coloring
mean_costs = sample_costs_viz.mean(axis=1)  # (K,)
# Normalize for colormap
cost_norm = (mean_costs - mean_costs.min()) / (mean_costs.max() - mean_costs.min() + 1e-10)

fig, ax = plt.subplots(figsize=(10, 10))

# Cost field
cost_field_viz = obstacle_cost_vectorized(sdf_data_viz)
ax.contourf(xc_viz, yc_viz, cost_field_viz, levels=20, cmap='hot_r', alpha=0.2)

# Obstacles
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.4)
    ax.add_patch(circle)
    ce = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='darkred', linewidth=2)
    ax.add_patch(ce)

# Plot noisy rollout EE paths (colored by cost: green=low, red=high)
cmap_rollout = plt.cm.RdYlGn_r
for k in range(K_viz):
    xi_noisy = xi_current + noise_viz[k]
    traj_noisy = full_trajectory(xi_noisy, q_start, q_goal)
    ee_noisy = np.array([forward_kinematics(q)[3] for q in traj_noisy])
    color = cmap_rollout(cost_norm[k])
    ax.plot(ee_noisy[:, 0], ee_noisy[:, 1], '-', color=color, alpha=0.4, linewidth=1)

# Current trajectory (thick navy)
traj_current = full_trajectory(xi_current, q_start, q_goal)
ee_current = np.array([forward_kinematics(q)[3] for q in traj_current])
ax.plot(ee_current[:, 0], ee_current[:, 1], '-', color='navy', linewidth=3, alpha=0.9, label='Current trajectory')

# Start and goal
plot_arm(ax, q_start, color='green', linewidth=3, markersize=8, label='Start')
plot_arm(ax, q_goal, color='red', linewidth=3, markersize=8, label='Goal')

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_rollout, norm=Normalize(vmin=mean_costs.min(), vmax=mean_costs.max()))
sm.set_array([])
plt.colorbar(sm, ax=ax, shrink=0.6, label='Mean rollout cost')

ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.0, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=11)
ax.set_title(f'STOMP Noisy Rollouts (K = {K_viz}): EE Paths Colored by Cost', fontsize=13)
plt.tight_layout()
plt.show()

print('Green rollouts have low cost -> high probability weight -> dominate the update.')
print('Red rollouts have high cost -> low probability weight -> mostly ignored.')

---
## 16. Parameter Analysis

In [ ]:
# K (number of samples) sensitivity
K_values = [5, 10, 20, 50]
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))

xi_init_k = init_trajectory(q_start, q_goal, N_WAYPOINTS)

for ax, K_val in zip(axes, K_values):
    np.random.seed(42)
    xi_opt_k, _, _ = stomp_with_obstacles(
        xi_init_k, q_start, q_goal, obstacles,
        K_samples=K_val, temperature=TEMPERATURE, lam=LAMBDA, max_iter=80)
    
    plot_trajectory_workspace(xi_opt_k, q_start, q_goal, ax=ax,
                              obstacles=obstacles,
                              title=f'K = {K_val} samples')

plt.suptitle('Effect of Number of Samples $K$', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('More samples (larger K) -> more robust optimization but slower per iteration.')
print('Fewer samples (smaller K) -> faster but noisier convergence.')

In [ ]:
# Temperature sensitivity: convergence curves
h_values = [0.1, 1.0, 10.0, 100.0]
colors_h = ['navy', 'steelblue', 'seagreen', 'coral']

fig, ax = plt.subplots(figsize=(10, 6))
xi_init_h = init_trajectory(q_start, q_goal, N_WAYPOINTS)

for h_val, color in zip(h_values, colors_h):
    np.random.seed(42)
    _, costs_h, _ = stomp_with_obstacles(
        xi_init_h, q_start, q_goal, obstacles,
        K_samples=K_SAMPLES, temperature=h_val, lam=LAMBDA, max_iter=100)
    ax.plot(costs_h['total'], color=color, linewidth=2, label=f'h = {h_val}')

ax.set_xlabel('Iteration')
ax.set_ylabel('Total Cost')
ax.set_title('Temperature Sensitivity: Cost Convergence')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('Low h (greedy): fast convergence but may get stuck in local minima.')
print('High h (uniform): slower convergence but more exploration.')

In [ ]:
# Temperature workspace comparison
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
xi_init_h = init_trajectory(q_start, q_goal, N_WAYPOINTS)

for ax, h_val in zip(axes, h_values):
    np.random.seed(42)
    xi_opt_h, _, _ = stomp_with_obstacles(
        xi_init_h, q_start, q_goal, obstacles,
        K_samples=K_SAMPLES, temperature=h_val, lam=LAMBDA, max_iter=100)
    
    plot_trajectory_workspace(xi_opt_h, q_start, q_goal, ax=ax,
                              obstacles=obstacles,
                              title=f'h = {h_val}')

plt.suptitle('Effect of Temperature $h$ on Optimized Trajectory', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Stochasticity: run STOMP 4 times with different seeds
seeds = [0, 7, 42, 99]
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))

xi_init_stoch = init_trajectory(q_start, q_goal, N_WAYPOINTS)

for ax, seed in zip(axes, seeds):
    np.random.seed(seed)
    xi_opt_s, _, _ = stomp_with_obstacles(
        xi_init_stoch, q_start, q_goal, obstacles,
        K_samples=K_SAMPLES, temperature=TEMPERATURE, lam=LAMBDA, max_iter=100)
    
    plot_trajectory_workspace(xi_opt_s, q_start, q_goal, ax=ax,
                              obstacles=obstacles,
                              title=f'Seed = {seed}')

plt.suptitle('Stochasticity: Different Random Seeds -> Different Solutions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("STOMP's stochasticity can find different local minima — an advantage over deterministic CHOMP.")

---
## 17. Limitations & Extensions

### Limitations
- **Noisy convergence:** Cost may not decrease monotonically due to stochastic updates
- **$K$ evaluations per iteration:** Each iteration requires $K$ full cost evaluations (vs. 1 for CHOMP)
- **Local optimizer:** Like CHOMP, STOMP finds local (not global) optima
- **Tuning $K$ and $h$:** Performance is sensitive to the number of samples and temperature

### Strengths Over CHOMP
- **No gradients needed:** Works with any cost function — non-differentiable, black-box, or noisy
- **More exploration:** Stochastic nature helps escape some local minima
- **Natural parallelism:** The $K$ rollouts are independent and embarrassingly parallel

### Comparison of Trajectory Optimizers

| | **CHOMP** | **STOMP** | **TrajOpt** | **GPMP2** |
|---|---|---|---|---|
| **Update** | Covariant gradient | Probability-weighted sampling | Sequential convex | Factor graph (MAP) |
| **Needs gradients?** | Yes | **No** | Yes | Yes |
| **Convergence** | Smooth, monotone | Noisy, stochastic | Fast (convex subproblems) | Fast (sparse solve) |
| **Exploration** | None (deterministic) | Stochastic | None | Prior-based |
| **Parallelism** | Low | **High** ($K$ rollouts) | Low | Medium |

### Key Takeaways
1. STOMP replaces gradient computation with **stochastic sampling** — conceptually simple, broadly applicable
2. The $R^{-1}$ matrix is central: it generates smooth noise AND smooths updates
3. Probability weighting acts as a **soft selection** mechanism among noisy rollouts
4. The $M$ matrix ensures updates remain smooth and bounded
5. STOMP's derivative-free nature makes it ideal for complex, non-differentiable cost landscapes